# LangGraph + PayGraph: Slack Approval via `interrupt()`

This notebook demonstrates the LangGraph interrupt integration added in issue #21. A PayGraph `AgentWallet` backed by `SlackApprovalGateway` is dropped into a LangGraph agent; when a spend needs human approval the graph pauses at the tool-call node via `interrupt()` (not a raw exception), persists the `request_id` in the checkpointer, and resumes via `Command(resume={"approved": True/False})`.

**Prerequisites**

```bash
pip install "paygraph[langgraph,live]"
```

and a real Slack incoming webhook URL (set below).

## 1. Configure the webhook

Replace `SLACK_WEBHOOK_URL` with your own incoming webhook. When the graph pauses you will see an approval message land in the channel.

In [ ]:
import os

SLACK_WEBHOOK_URL = os.environ.get(
    "SLACK_WEBHOOK_URL",
    "ADD_SLACK_WEBHOOK_URL",
)

## 2. Build the wallet

A `SpendPolicy` with `require_human_approval_above=20.0` means any spend over $20 triggers the Slack approval path. The inner gateway is `MockGateway(auto_approve=True)` so no real cards are minted.

In [ ]:
from paygraph import AgentWallet, SpendPolicy
from paygraph.gateways.mock import MockGateway
from paygraph.gateways.slack import SlackApprovalGateway

wallet = AgentWallet(
    gateways={
        "default": SlackApprovalGateway(
            webhook_url=SLACK_WEBHOOK_URL,
            inner_gateway=MockGateway(auto_approve=True),
        ),
    },
    policy=SpendPolicy(
        max_transaction=100.0,
        daily_budget=500.0,
        require_human_approval_above=20.0,
    ),
    verbose=False,
)

## 3. Build a minimal LangGraph graph using the spend tool

`wallet.langgraph_spend_tool` is the interrupt-aware counterpart to `wallet.spend_tool`. Here we wrap it in a trivial one-node graph so the example focuses on the pause/resume flow. In a real agent you'd register it with `create_react_agent(llm, tools=[wallet.langgraph_spend_tool])` instead.

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


class SpendState(TypedDict, total=False):
    amount: float
    vendor: str
    justification: str
    result: str


tool = wallet.langgraph_spend_tool


def spend_node(state: SpendState) -> dict:
    result = tool.invoke(
        {
            "amount": state["amount"],
            "vendor": state["vendor"],
            "justification": state["justification"],
        }
    )
    return {"result": result}


builder = StateGraph(SpendState)
builder.add_node("spend", spend_node)
builder.add_edge(START, "spend")
builder.add_edge("spend", END)

graph = builder.compile(checkpointer=InMemorySaver())

## 4. Trigger a spend that requires approval

Amount of $50 exceeds the $20 approval threshold. Running the graph:

1. Calls `wallet.request_spend(...)`
2. Posts an approval message to the Slack channel via the webhook
3. Pauses with `interrupt()` — the `request_id` is now in the checkpointed state
4. Returns a state object with an `__interrupt__` list describing the pause

Check Slack for the approval message. The `request_id` shown there matches the one in the returned state.

In [ ]:
config = {"configurable": {"thread_id": "notebook-demo-1"}}

state = graph.invoke(
    {
        "amount": 50.0,
        "vendor": "Anthropic",
        "justification": "need tokens for research",
    },
    config=config,
)

interrupts = state.get("__interrupt__", [])
assert interrupts, "expected the graph to pause for approval"
print("Paused with payload:")
print(interrupts[0].value)

## 5. (Optional) Cold-start resume

The checkpointer persists the interrupt payload. You can drop this notebook kernel, rebuild the graph with the same `InMemorySaver` (or, in production, a durable checkpointer like `SqliteSaver` or `PostgresSaver`), and read the pending state back using only the `thread_id`:

In [ ]:
snapshot = graph.get_state(config)
pending = snapshot.tasks[0].interrupts[0].value
print("Recovered from checkpoint:", pending)

## 6. Resume with the human's decision

In a production setup the webhook listener (sibling issue) would invoke this line after a human clicks **Approve**. For the notebook we pass it directly.

In [ ]:
from langgraph.types import Command

final = graph.invoke(Command(resume={"approved": True}), config=config)
print(final["result"])

## 7. Resume a different thread with a denial

The same graph handles denial cleanly: the tool returns a `Spend denied: ...` string and the audit log records a `denied` entry.

In [ ]:
deny_config = {"configurable": {"thread_id": "notebook-demo-deny"}}

graph.invoke(
    {"amount": 75.0, "vendor": "Acme", "justification": "risky purchase"},
    config=deny_config,
)
denied = graph.invoke(Command(resume={"approved": False}), config=deny_config)
print(denied["result"])

## Caveat: node re-run on resume

LangGraph re-runs the node body from the top after `interrupt()`, which means `wallet.request_spend()` is called a second time when the graph resumes. The second call posts a fresh Slack message with a new `request_id`; the `interrupt()` then returns the human's decision immediately and the **fresh** `request_id` is the one that ultimately gets completed. The first `request_id` is orphaned (not charged).

For strict single-post semantics, split the flow into two nodes and use `interrupt_before` on the second — the first persists the `request_id` into graph state, and the second reads it back on resume.